# Assignment 1 — training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file — `main.py` refuses to run
if it finds both `train.py` and `train.ipynb`, so exactly one of them must
exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py train`, this notebook must, start to finish, with
no manual steps in between:

- Load your committed dataset.
- Apply the feature engineering, preprocessing, and train/val/test split
  consistent with the business framing you wrote in `REPORT.md`.
- Train the same model — linear regression for a regression target,
  logistic regression for a binary one — three ways, on the same split:
  scikit-learn, a from-scratch PyTorch loop (Session 5's manual approach:
  raw tensors, autograd, a manual gradient-step update), and the standard
  `torch.nn.Module` + `torch.optim` workflow.
- Evaluate all three against each other and against a naive baseline.
- Save whatever `app.py`/`app.ipynb` needs to build its dashboard without
  retraining anything.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

In [3]:
import pandas as pd
import numpy as np

RAW_DATA_PATH = "data/airline_satisfaction.csv"

df = pd.read_csv(RAW_DATA_PATH, parse_dates=["Flight Date"])

df.columns = df.columns.str.strip()

print(f"shape: {df.shape}")
df.info()

shape: (10600, 22)
<class 'pandas.DataFrame'>
RangeIndex: 10600 entries, 0 to 10599
Data columns (total 22 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   CustomerID                              10600 non-null  int64         
 1   Satisfaction                            10600 non-null  str           
 2   Airline Status                          10600 non-null  str           
 3   Age                                     10600 non-null  int64         
 4   Gender                                  10600 non-null  str           
 5   Type of Travel                          10600 non-null  str           
 6   Shopping Amount at Airport              10600 non-null  int64         
 7   Eating and Drinking Amounts at Airport  10600 non-null  int64         
 8   Class                                   10600 non-null  str           
 9   Flight Date                             10

In [4]:
# Duplicates
print("Duplicate rows:", df.duplicated().sum())

# Missing values
na_counts = df.isna().sum()
print("\nColumns with missing values:")
print(na_counts[na_counts > 0] if na_counts.any() else "none")

# Target distribution
print("\nTarget distribution:")
print(df["Satisfaction"].value_counts())
print(df["Satisfaction"].value_counts(normalize=True).round(3))

# Date range (relevant for the time-based split decided in REPORT.md)
print("\nFlight Date range:", df["Flight Date"].min(), "to", df["Flight Date"].max())

Duplicate rows: 0

Columns with missing values:
Flight Time        418
Arrival Delay      418
Departure Delay    379
dtype: int64

Target distribution:
Satisfaction
Satisfied        5430
Not Satisfied    5170
Name: count, dtype: int64
Satisfaction
Satisfied        0.512
Not Satisfied    0.488
Name: proportion, dtype: float64

Flight Date range: 2019-01-01 00:00:00 to 2019-12-31 00:00:00


In [5]:
# Not Satisfied = 1
df["target"] = (df["Satisfaction"] == "Not Satisfied").astype(int)

print(df["target"].value_counts())
print(df[["Satisfaction", "target"]].drop_duplicates())

target
0    5430
1    5170
Name: count, dtype: int64
    Satisfaction  target
0      Satisfied       0
7  Not Satisfied       1


In [6]:
# Identify rows with missing delay or time data
missing_delay_mask = df["Flight Time"].isna() | df["Arrival Delay"].isna() | df["Departure Delay"].isna()

print("Rows with missing delay/time data:", missing_delay_mask.sum())
print("\nFlight Cancelled breakdown for those rows:")
print(df.loc[missing_delay_mask, "Flight Cancelled"].value_counts())

print("\nFlight Cancelled breakdown overall:")
print(df["Flight Cancelled"].value_counts())

Rows with missing delay/time data: 418

Flight Cancelled breakdown for those rows:
Flight Cancelled
Yes    393
No      25
Name: count, dtype: int64

Flight Cancelled breakdown overall:
Flight Cancelled
No     10207
Yes      393
Name: count, dtype: int64


In [7]:
# Confirming the pattern
# Cancelled flights
cancelled = df[df["Flight Cancelled"] == "Yes"]
print("Cancelled flights missing Flight Time:", cancelled["Flight Time"].isna().sum(), "/", len(cancelled))
print("Cancelled flights missing Arrival Delay:", cancelled["Arrival Delay"].isna().sum(), "/", len(cancelled))
print("Cancelled flights missing Departure Delay:", cancelled["Departure Delay"].isna().sum(), "/", len(cancelled))

# Missing but NOT cancelled
anomaly = df[(df["Flight Cancelled"] == "No") & missing_delay_mask]
print("\nAnomalous rows (not cancelled but missing data):")
print(anomaly[["Flight Time", "Arrival Delay", "Departure Delay"]])

Cancelled flights missing Flight Time: 393 / 393
Cancelled flights missing Arrival Delay: 393 / 393
Cancelled flights missing Departure Delay: 379 / 393

Anomalous rows (not cancelled but missing data):
       Flight Time  Arrival Delay  Departure Delay
85             NaN            NaN              0.0
193            NaN            NaN            266.0
429            NaN            NaN              6.0
437            NaN            NaN            118.0
938            NaN            NaN            109.0
1491           NaN            NaN              0.0
1553           NaN            NaN              0.0
2701           NaN            NaN             70.0
3155           NaN            NaN              0.0
3262           NaN            NaN              0.0
3285           NaN            NaN              0.0
3337           NaN            NaN              5.0
3439           NaN            NaN              0.0
3490           NaN            NaN              0.0
3914           NaN            Na

In [8]:
# Create a new column to indicate whether the flight was completed (not cancelled and not missing both Flight Time and Arrival Delay)
df["flight_completed"] = (~(df["Flight Time"].isna() & df["Arrival Delay"].isna())).astype(int)

# Fill missing values for Flight Time, Arrival Delay, and Departure Delay with 0
df["Flight Time"] = df["Flight Time"].fillna(0)
df["Arrival Delay"] = df["Arrival Delay"].fillna(0)
df["Departure Delay"] = df["Departure Delay"].fillna(0)

# Check
print(df[["Flight Time", "Arrival Delay", "Departure Delay", "flight_completed"]].isna().sum())

Flight Time         0
Arrival Delay       0
Departure Delay     0
flight_completed    0
dtype: int64


In [9]:
# Ordinal
status_order = {"Blue": 0, "Silver": 1, "Gold": 2, "Platinum": 3}
df["Airline_Status_Ordinal"] = df["Airline Status"].map(status_order)

class_order = {"Eco": 0, "Eco Plus": 1, "Business": 2}
df["Class_Ordinal"] = df["Class"].map(class_order)

# One-hot
df = pd.get_dummies(df, columns=["Gender", "Type of Travel", "Airline Name"], 
                     prefix=["Gender", "Travel", "Airline"], drop_first=True)

print(df[["Airline_Status_Ordinal", "Class_Ordinal"]].head())
print("\nNew columns after one-hot:")
print([c for c in df.columns if c.startswith(("Gender_", "Travel_", "Airline_"))])

   Airline_Status_Ordinal  Class_Ordinal
0                       1              0
1                       0              0
2                       1              0
3                       0              0
4                       0              0

New columns after one-hot:
['Airline_Status_Ordinal', 'Gender_Male', 'Travel_Mileage tickets', 'Travel_Personal Travel', 'Airline_Wow Air']


In [10]:
# Mapping state -> region
NORTHEAST = {"Connecticut","Maine","Massachusetts","New Hampshire","Rhode Island","Vermont",
             "New Jersey","New York","Pennsylvania"}
MIDWEST = {"Illinois","Indiana","Michigan","Ohio","Wisconsin","Iowa","Kansas","Minnesota",
           "Missouri","Nebraska","North Dakota","South Dakota"}
SOUTH = {"Delaware","Florida","Georgia","Maryland","North Carolina","South Carolina",
         "Virginia","West Virginia","Alabama","Kentucky","Mississippi","Tennessee",
         "Arkansas","Louisiana","Oklahoma","Texas"}
WEST = {"Arizona","Colorado","Idaho","Montana","Nevada","New Mexico","Utah","Wyoming",
        "Alaska","California","Hawaii","Oregon","Washington"}

def state_to_region(state):
    if state in NORTHEAST: return "Northeast"
    if state in MIDWEST: return "Midwest"
    if state in SOUTH: return "South"
    if state in WEST: return "West"
    return "Other"  # catch-all in case a state name doesn't match / typo

df["Origin_Region"] = df["Origin State"].apply(state_to_region)
df["Destination_Region"] = df["Destination State"].apply(state_to_region)

print("Origin region distribution:")
print(df["Origin_Region"].value_counts())
print("\nDestination region distribution:")
print(df["Destination_Region"].value_counts())

Origin region distribution:
Origin_Region
South        6360
Midwest      3146
Northeast     758
West          336
Name: count, dtype: int64

Destination region distribution:
Destination_Region
South        5979
Midwest      3246
Northeast    1054
West          321
Name: count, dtype: int64


In [11]:
# One-hot encode the region columns and drop the original state columns
df = pd.get_dummies(df, columns=["Origin_Region", "Destination_Region"],
                     prefix=["OriginReg", "DestReg"], drop_first=True)

columns_to_drop = [
    "CustomerID",            # just an identifier, not a predictive feature
    "Satisfaction",          # already replaced by the "target" column
    "Airline Status",        # already replaced by Airline_Status_Ordinal
    "Class",                 # already replaced by Class_Ordinal
    "Origin State",          # already replaced by OriginReg_*
    "Destination State",     # already replaced by DestReg_*
    "Flight Cancelled",      # already replaced by flight_completed (more accurate -- catches the 25 anomaly rows too)
    "Month of Flight Date",  # redundant -- derivable from Flight Date if needed
]
df = df.drop(columns=columns_to_drop)

print("Remaining columns:")
print(df.columns.tolist())
print("\nShape now:", df.shape)

Remaining columns:
['Age', 'Shopping Amount at Airport', 'Eating and Drinking Amounts at Airport', 'Flight Date', 'Day of Flight Date', 'No of Flights', 'Flight Time', 'Arrival Delay', 'Departure Delay', 'Scheduled Departure Hour', 'Flight Distance', 'target', 'flight_completed', 'Airline_Status_Ordinal', 'Class_Ordinal', 'Gender_Male', 'Travel_Mileage tickets', 'Travel_Personal Travel', 'Airline_Wow Air', 'OriginReg_Northeast', 'OriginReg_South', 'OriginReg_West', 'DestReg_Northeast', 'DestReg_South', 'DestReg_West']

Shape now: (10600, 25)


In [12]:
# Time-based split: 80% train, 20% test, based on Flight Date (chronological order)
df_sorted = df.sort_values("Flight Date").reset_index(drop=True)

split_idx = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_idx].copy()
test_df = df_sorted.iloc[split_idx:].copy()

print("Train:", train_df.shape, train_df["Flight Date"].min(), "to", train_df["Flight Date"].max())
print("Test:", test_df.shape, test_df["Flight Date"].min(), "to", test_df["Flight Date"].max())
print("\nTrain target balance:", train_df["target"].mean().round(3))
print("Test target balance:", test_df["target"].mean().round(3))

Train: (8480, 25) 2019-01-01 00:00:00 to 2019-11-02 00:00:00
Test: (2120, 25) 2019-11-02 00:00:00 to 2019-12-31 00:00:00

Train target balance: 0.49
Test target balance: 0.48


In [13]:
# Feature scaling
from sklearn.preprocessing import StandardScaler

FEATURE_COLS = [c for c in df_sorted.columns if c not in ("target", "Flight Date")]

X_train_raw = train_df[FEATURE_COLS].values.astype(np.float64)
X_test_raw = test_df[FEATURE_COLS].values.astype(np.float64)
y_train = train_df["target"].values.astype(np.float64)
y_test = test_df["target"].values.astype(np.float64)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print("Feature count:", len(FEATURE_COLS))
print(FEATURE_COLS)
print("\nX_train shape:", X_train.shape, "X_test shape:", X_test.shape)

Feature count: 23
['Age', 'Shopping Amount at Airport', 'Eating and Drinking Amounts at Airport', 'Day of Flight Date', 'No of Flights', 'Flight Time', 'Arrival Delay', 'Departure Delay', 'Scheduled Departure Hour', 'Flight Distance', 'flight_completed', 'Airline_Status_Ordinal', 'Class_Ordinal', 'Gender_Male', 'Travel_Mileage tickets', 'Travel_Personal Travel', 'Airline_Wow Air', 'OriginReg_Northeast', 'OriginReg_South', 'OriginReg_West', 'DestReg_Northeast', 'DestReg_South', 'DestReg_West']

X_train shape: (8480, 23) X_test shape: (2120, 23)


In [14]:
# Baseline model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

majority_class = 1 if y_train.mean() > 0.5 else 0
baseline_preds = np.full_like(y_test, fill_value=majority_class)

def evaluate(y_true, y_pred, name):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

baseline_metrics = evaluate(y_test, baseline_preds, "Naive baseline (majority class)")
print(baseline_metrics)

{'model': 'Naive baseline (majority class)', 'accuracy': 0.519811320754717, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}


In [15]:
# Model 1: Sckit-learn logistic regression
from sklearn.linear_model import LogisticRegression

SKLEARN_C = 1.0

sklearn_model = LogisticRegression(C=SKLEARN_C, max_iter=1000)
sklearn_model.fit(X_train, y_train)

sklearn_probs = sklearn_model.predict_proba(X_test)[:, 1]
sklearn_preds = (sklearn_probs >= 0.5).astype(int)

sklearn_metrics = evaluate(y_test, sklearn_preds, "scikit-learn")
print(sklearn_metrics)

{'model': 'scikit-learn', 'accuracy': 0.7613207547169811, 'precision': 0.8626062322946175, 'recall': 0.5982318271119843, 'f1': 0.7064965197215777}


In [ ]:
# C check
train_preds_check = (sklearn_model.predict_proba(X_train)[:, 1] >= 0.5).astype(int)
print("Train accuracy:", accuracy_score(y_train, train_preds_check))
print("Test accuracy:", accuracy_score(y_test, sklearn_preds))

Train accuracy: 0.769811320754717
Test accuracy: 0.7613207547169811


In [17]:
# Model 2: Manual PyTorch
import torch

torch.manual_seed(42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

n_features = X_train_t.shape[1]
w = torch.zeros(n_features, dtype=torch.float32, requires_grad=True)
b = torch.zeros(1, dtype=torch.float32, requires_grad=True)

LEARNING_RATE = 0.1
N_EPOCHS = 2000
L2_LAMBDA = 1.0 / (SKLEARN_C * len(X_train_t))

for epoch in range(N_EPOCHS):
    logits = X_train_t @ w + b
    probs = torch.sigmoid(logits)

    # binary cross-entropy loss
    bce = -(y_train_t * torch.log(probs + 1e-8) + (1 - y_train_t) * torch.log(1 - probs + 1e-8)).mean()
    loss = bce + L2_LAMBDA * (w ** 2).sum()

    # gradient
    loss.backward()

    with torch.no_grad():
        w -= LEARNING_RATE * w.grad
        b -= LEARNING_RATE * b.grad
        w.grad.zero_()
        b.grad.zero_()

    if epoch % 500 == 0:
        print(f"epoch {epoch}: loss={loss.item():.4f}")

with torch.no_grad():
    manual_pt_probs = torch.sigmoid(X_test_t @ w + b).numpy()
manual_pt_preds = (manual_pt_probs >= 0.5).astype(int)

manual_pt_metrics = evaluate(y_test, manual_pt_preds, "Manual PyTorch loop")
print(manual_pt_metrics)

epoch 0: loss=0.6931
epoch 500: loss=0.4938
epoch 1000: loss=0.4938
epoch 1500: loss=0.4938
{'model': 'Manual PyTorch loop', 'accuracy': 0.7617924528301887, 'precision': 0.8628005657708628, 'recall': 0.5992141453831041, 'f1': 0.7072463768115942}


In [18]:
# Model 3: Standard PyTorch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

class LogisticRegressionModule(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(1)

standard_model = LogisticRegressionModule(n_features)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(standard_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_LAMBDA)

for epoch in range(N_EPOCHS):
    optimizer.zero_grad()
    logits = standard_model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f"epoch {epoch}: loss={loss.item():.4f}")

standard_model.eval()
with torch.no_grad():
    standard_pt_probs = torch.sigmoid(standard_model(X_test_t)).numpy()
standard_pt_preds = (standard_pt_probs >= 0.5).astype(int)

standard_pt_metrics = evaluate(y_test, standard_pt_preds, "Standard PyTorch (nn.Module)")
print(standard_pt_metrics)

epoch 0: loss=0.7516
epoch 500: loss=0.4937
epoch 1000: loss=0.4935
epoch 1500: loss=0.4935
{'model': 'Standard PyTorch (nn.Module)', 'accuracy': 0.7603773584905661, 'precision': 0.8611898016997167, 'recall': 0.5972495088408645, 'f1': 0.7053364269141531}


In [19]:
# Comparison of all models
comparison_df = pd.DataFrame([baseline_metrics, sklearn_metrics, manual_pt_metrics, standard_pt_metrics])
comparison_df = comparison_df.set_index("model").round(4)
print(comparison_df)

sklearn_coefs = sklearn_model.coef_.flatten()
manual_pt_coefs = w.detach().numpy()
standard_pt_coefs = standard_model.linear.weight.detach().numpy().flatten()

coef_comparison = pd.DataFrame({
    "feature": FEATURE_COLS,
    "sklearn": sklearn_coefs,
    "manual_pytorch": manual_pt_coefs,
    "standard_pytorch": standard_pt_coefs,
}).round(3)
print("\nCoefficient comparison (first 10 features):")
print(coef_comparison.head(10))

                                 accuracy  precision  recall      f1
model                                                               
Naive baseline (majority class)    0.5198     0.0000  0.0000  0.0000
scikit-learn                       0.7613     0.8626  0.5982  0.7065
Manual PyTorch loop                0.7618     0.8628  0.5992  0.7072
Standard PyTorch (nn.Module)       0.7604     0.8612  0.5972  0.7053

Coefficient comparison (first 10 features):
                                  feature  sklearn  manual_pytorch  \
0                                     Age    0.053           0.053   
1              Shopping Amount at Airport   -0.086          -0.086   
2  Eating and Drinking Amounts at Airport   -0.018          -0.019   
3                      Day of Flight Date   -0.059          -0.059   
4                           No of Flights    0.256           0.256   
5                             Flight Time    0.179           0.150   
6                           Arrival Delay    0.212 

In [20]:
# Save artifacts for the app to use
import joblib
import json
from pathlib import Path

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

joblib.dump(sklearn_model, ARTIFACTS_DIR / "sklearn_model.joblib")
joblib.dump(scaler, ARTIFACTS_DIR / "scaler.joblib")

torch.save({"w": w.detach(), "b": b.detach()}, ARTIFACTS_DIR / "manual_pytorch_weights.pt")

torch.save(
    {"state_dict": standard_model.state_dict(), "n_features": n_features},
    ARTIFACTS_DIR / "standard_pytorch_model.pt",
)

with open(ARTIFACTS_DIR / "feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f)

test_export = test_df[FEATURE_COLS + ["target", "Flight Date"]].copy()
test_export.to_csv(ARTIFACTS_DIR / "test_data.csv", index=False)

comparison_df.to_csv(ARTIFACTS_DIR / "comparison_metrics.csv")

print("Saved to", ARTIFACTS_DIR.resolve())
print(list(ARTIFACTS_DIR.iterdir()))

Saved to C:\Users\sophi\Documents\Designing AI\daiist-assignment-1\artifacts
[WindowsPath('artifacts/comparison_metrics.csv'), WindowsPath('artifacts/feature_cols.json'), WindowsPath('artifacts/manual_pytorch_weights.pt'), WindowsPath('artifacts/scaler.joblib'), WindowsPath('artifacts/sklearn_model.joblib'), WindowsPath('artifacts/standard_pytorch_model.pt'), WindowsPath('artifacts/test_data.csv')]
